# 10 · Appendix numbers and figures for the midterm report

Recomputes the numbers Section 5 quotes from the pair-level margins and draws the three appendix figures.
Run it on CARC (it reads `artifacts/runs` and `data/v2`; nothing is trained). Nothing is saved until you run the
last cell, which writes the tables (CSV) and figures (PNG + PDF) to `artifacts/analysis/appendix/`.

| Output | Backs this claim in Section 5 |
|---|---|
| Table A1, A2, A3 and Figure A1 | Safety prefers short answers; old pairs that prefer the shorter answer are forgotten far more, most clearly during the Quality stage |
| Table A4, A5 and Figure A2 | MFR and Lowest Margin replay different training pairs but lose mostly the same validation pairs |
| Table A6 and Figure A3 | Forgetting is concentrated in weakly learned pairs; Lowest Margin recovers more than MFR |

## Settings, data, and helpers

In [ ]:
import json, os
from IPython.display import display
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get('G26', '/project2/xiangren_1987/group26'))
RUNS = Path(os.environ.get('G26_RUNS', ROOT / 'artifacts' / 'runs'))
DATA = Path(os.environ.get('G26_DATA', ROOT / 'data' / 'v2'))
OUT = Path(os.environ.get('G26_OUT', ROOT / 'artifacts' / 'analysis' / 'appendix'))
METHODS = ['none', 'random', 'random_high', 'mfr', 'lowest_margin']
LABEL = {'none': 'No replay', 'random': 'Random 10%', 'random_high': 'Random 14.3%', 'mfr': 'MFR',
         'lowest_margin': 'Lowest Margin'}
DL = {'helpful': 'Helpful', 'safe': 'Safe', 'quality': 'Quality'}
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

runs = {}
for f in RUNS.glob('*/settings.json'):
    d = f.parent
    if not (d / 'COMPLETE.json').exists():
        continue
    s = json.loads(f.read_text())
    if s['method'] in METHODS and int(s['order_id']) in (1, 2, 3, 4) and int(s['seed']) in (0, 1):
        runs[(s['method'], int(s['order_id']), int(s['seed']))] = (d, s)
print('runs per method:', {m: sum(k[0] == m for k in runs) for m in METHODS})


def sdir(d, s, k):
    ds = s['order'][k - 1]
    p = d / f'stage{k}_{ds}'
    if p.exists() or k != 1 or not s.get('stage1_from'):
        return p
    src = Path(s['stage1_from'])
    return (src if src.exists() else d.parent / src.name) / f'stage1_{ds}'


def M(d, s, k, ds):
    return pd.read_csv(sdir(d, s, k) / f'margins_{ds}_val.csv').set_index('id')['margin']


gap = {}
for ds in DL:
    with open(DATA / f'{ds}_val.jsonl') as fh:
        for line in fh:
            r = json.loads(line)
            gap[r['id']] = r['chosen_tokens'] - r['rejected_tokens']
gap = pd.Series(gap)


TABLES, FIGURES = {}, {}

def show(df, name, title):
    print(f'\n=== {name}: {title}')
    display(df.round(2))
    TABLES[name] = df

## Plot style

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
MCOLOR = {'none': '#8a8984', 'random': '#2a78d6', 'random_high': '#86b6ef', 'mfr': '#eb6834', 'lowest_margin': '#1baf7a'}
SHORT_C, LONG_C = '#e34948', '#8fa8c8'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e5e4e0'
SEQ = LinearSegmentedColormap.from_list('seq', ['#f4f8fe', '#86b6ef', '#2a78d6', '#184f95'])
FULL_W = 6.5
def style(font='Carlito'):
    from matplotlib import font_manager as fm
    names = {f.name for f in fm.fontManager.ttflist}
    family = next((n for n in ['Calibri', font] if n in names), 'DejaVu Sans')
    plt.rcParams.update({
        'font.family': family, 'font.size': 9.5, 'axes.titlesize': 9.5, 'axes.titleweight': 'bold',
        'axes.titlelocation': 'left', 'axes.labelsize': 9.5, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
        'legend.fontsize': 8.5, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.edgecolor': '#8a8984', 'axes.linewidth': 0.8, 'ytick.major.size': 0, 'xtick.major.size': 3,
        'figure.dpi': 150, 'savefig.dpi': 300, 'figure.constrained_layout.use': True, 'pdf.fonttype': 42,
        'figure.facecolor': 'white'})
    return family


def _bars(ax, groups, short_vals, long_vals, ylabel):
    x = np.arange(len(groups))
    w = 0.36
    for dx, vals, c, lab in [(-w / 2, short_vals, SHORT_C, 'Preferred answer is shorter'),
                             (w / 2, long_vals, LONG_C, 'Preferred answer is longer')]:
        b = ax.bar(x + dx, vals, width=w, color=c, label=lab, zorder=2)
        for xi, v in zip(x + dx, vals):
            ax.annotate(f'{v:.1f}', (xi, v), xytext=(0, 2), textcoords='offset points', ha='center', fontsize=8)
    ax.set_xticks(x, groups)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', color=GRID, lw=0.7)
    ax.set_axisbelow(True)


def fig_length(a2, a3):
    """a2: rows = stage trained (Helpful, Safe, Quality) with columns short_flip, long_flip.
    a3: rows = ['Quality not after Safety', 'Quality after Safety'] with columns short_forgot, long_forgot."""
    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.9), width_ratios=[3, 2])
    _bars(axes[0], [f'{s} stage' for s in a2.index], a2['short_flip'], a2['long_flip'],
          'Old pairs flipped to wrong (%)')
    axes[0].set_title('a   During each new stage (all old pairs)')
    _bars(axes[1], ['Quality not after\n(order 4)', 'Quality after\n(orders 1, 2)'], a3['short_forgot'],
          a3['long_forgot'], 'Safety pairs forgotten (%)')
    axes[1].set_title('b   Safety pairs by the end')
    top = max(a2[['short_flip', 'long_flip']].max().max(), a3[['short_forgot', 'long_forgot']].max().max()) * 1.15
    for ax in axes:
        ax.set_ylim(0, top)
    fig.legend(*axes[0].get_legend_handles_labels(), loc='outside upper center', ncol=2)
    return fig


def _heat(ax, mat, rows, cols, fmt, vmax, skip_diag=True):
    m = np.array(mat, float)
    show = m.copy()
    if skip_diag:
        for i in range(min(m.shape)):
            show[i, i] = np.nan
    ax.imshow(show, cmap=SEQ, vmin=0, vmax=vmax, aspect='auto')
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if skip_diag and i == j:
                continue
            v = m[i, j]
            ax.text(j, i, fmt(v), ha='center', va='center', fontsize=8.5,
                    color='white' if v > 0.6 * vmax else INK)
    ax.set_xticks(range(len(cols)), cols, rotation=30, ha='right', rotation_mode='anchor')
    ax.set_yticks(range(len(rows)), rows)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)


def fig_overlap(a5, a4):
    """a5: 4x4 DataFrame (replay methods) of Jaccard; a4: 5x5 DataFrame of % also forgotten (rows forgotten by)."""
    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 3.1), width_ratios=[4, 5])
    rl = [LABEL[m] for m in a5.index]
    _heat(axes[0], a5.values, rl, rl, lambda v: f'{v:.2f}', 1.0)
    axes[0].set_title('a   Training pairs replayed\n     (overlap, 0 to 1)')
    fl = [LABEL[m] for m in a4.index]
    _heat(axes[1], a4.values, fl, fl, lambda v: f'{v:.0f}%', 100.0)
    axes[1].set_title('b   Of pairs the row method forgets,\n     % also forgotten by the column method')
    return fig


def fig_margin(a6):
    """a6: rows = methods, columns Q1..Q4, forget, recovery."""
    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, 2.8), width_ratios=[3, 2])
    for m in a6.index:
        axes[0].plot([1, 2, 3, 4], a6.loc[m, ['Q1', 'Q2', 'Q3', 'Q4']], marker='o', ms=4, lw=1.8,
                     color=MCOLOR[m], label=LABEL[m])
    axes[0].set_xticks([1, 2, 3, 4], ['Q1\nweakest', 'Q2', 'Q3', 'Q4\nstrongest'])
    axes[0].set_ylabel('Forgotten by the end (%)')
    axes[0].set_title('a   By how strongly a pair was learned')
    axes[0].grid(axis='y', color=GRID, lw=0.7)
    y = np.arange(len(a6.index))
    h = 0.38
    axes[1].barh(y - h / 2, a6['forget'], height=h, color='#e34948', label='Forgotten (% of learned)')
    axes[1].barh(y + h / 2, a6['recovery'], height=h, color='#2a78d6', label='Recovered (% of not learned)')
    for yi, (f, r) in enumerate(zip(a6['forget'], a6['recovery'])):
        axes[1].annotate(f'{f:.1f}', (f, yi - h / 2), xytext=(2, 0), textcoords='offset points', va='center', fontsize=7.5)
        axes[1].annotate(f'{r:.1f}', (r, yi + h / 2), xytext=(2, 0), textcoords='offset points', va='center', fontsize=7.5)
    axes[1].set_yticks(y, [LABEL[m] for m in a6.index])
    axes[1].set_xlim(0, max(a6['forget'].max(), a6['recovery'].max()) * 1.25)
    axes[1].set_ylim(len(y) - 0.5, -1.9)
    axes[1].set_title('b   Forgotten vs recovered')
    axes[1].grid(axis='x', color=GRID, lw=0.7)
    axes[1].tick_params(axis='y', length=0)
    axes[0].set_ylim(0, None)
    axes[0].legend(loc='upper right', fontsize=8)
    axes[1].legend(loc='upper right', fontsize=7.5, borderaxespad=0.1)
    return fig

def _load_report_font():
    from matplotlib import font_manager as fm
    cache = Path.home() / '.cache' / 'mfr_fonts'
    for f in cache.glob('*.ttf'):
        fm.fontManager.addfont(str(f))
_load_report_font()
print('figure font:', style())


## Table A1. Is the preferred answer longer or shorter?

In [ ]:
# A1. Preferred vs rejected length per behavior (validation pairs)
rows = []
for ds in DL:
    g = gap[gap.index.str.startswith(ds)]
    rows.append({'Behavior': DL[ds], 'Median chosen - rejected (tokens)': g.median(),
                 'Pairs where chosen is shorter (%)': 100 * (g < 0).mean(), 'Pairs': len(g)})
show(pd.DataFrame(rows), 'A1_length_by_behavior', 'Is the preferred answer longer or shorter than the rejected one?')

## Table A2. What each new stage does to old pairs, by answer length (No replay)

In [ ]:
# A2. What each new stage does to old pairs, split by whether the old pair prefers the shorter answer (No replay)
rows = []
for (m, o, sd), (d, s) in runs.items():
    if m != 'none':
        continue
    order = s['order']
    for k in range(2, len(order) + 1):
        new = order[k - 1]
        for old in order[:k - 1]:
            before, after = M(d, s, k - 1, old).align(M(d, s, k, old), join='inner')
            g = gap.reindex(before.index)
            for short, mask in [(True, g < 0), (False, g > 0)]:
                b, a = before[mask], after[mask]
                known = b > 0
                rows.append({'new_stage': new, 'old': old, 'short': short, 'n': len(b),
                             'margin_change_x1000': 1000 * (a - b).mean(),
                             'flipped_pct': 100 * (known & (a <= 0)).sum() / max(known.sum(), 1)})
t = pd.DataFrame(rows)
a2 = (t.groupby(['new_stage', 'short'])[['margin_change_x1000', 'flipped_pct']].mean().unstack('short'))
a2.columns = [f'{"chosen shorter" if sh else "chosen longer"}: {c}' for c, sh in a2.columns]
a2 = a2.reset_index().rename(columns={'new_stage': 'Stage being trained'})
a2['Stage being trained'] = a2['Stage being trained'].map(DL)
show(a2, 'A2_stage_effect_by_length',
     'No replay: change in old pairs during each new stage, by whether the old pair prefers the shorter answer '
     '(margin change x1000; flipped = % of correct old pairs that become wrong during that stage)')

## Table A3. Safety forgetting with and without a later Quality stage (No replay)

In [ ]:
# A3. Safety forgetting depending on whether Quality is trained after Safety (No replay)
rows = []
for (m, o, sd), (d, s) in runs.items():
    if m != 'none' or 'safe' not in s['order'][:-1]:
        continue
    order, last = s['order'], len(s['order'])
    k = order.index('safe') + 1
    a, z = M(d, s, k, 'safe').align(M(d, s, last, 'safe'), join='inner')
    g = gap.reindex(a.index)
    after_quality = 'quality' in order[k:]
    for short, mask in [(True, g < 0), (False, g > 0)]:
        kn = a[mask] > 0
        rows.append({'order': o, 'seed': sd, 'Quality after Safety': after_quality, 'short': short,
                     'forgot_pct': 100 * (kn & (z[mask] <= 0)).sum() / max(kn.sum(), 1),
                     'acc_change': 100 * ((z[mask] > 0).mean() - (a[mask] > 0).mean())})
t = pd.DataFrame(rows)
a3 = t.groupby(['Quality after Safety', 'short'])[['forgot_pct', 'acc_change']].mean().reset_index()
a3['Safety pairs'] = np.where(a3['short'], 'chosen shorter', 'chosen longer')
a3 = a3[['Quality after Safety', 'Safety pairs', 'forgot_pct', 'acc_change']].rename(
    columns={'forgot_pct': 'Forgotten (% of learned)', 'acc_change': 'Accuracy change (points)'})
show(a3, 'A3_safety_vs_quality_after', 'No replay: Safety forgetting when Quality comes after Safety (orders 1, 2) vs not (order 4)')

## Table A4. Do methods forget the same pairs?

In [ ]:
# A4. Of the pairs a method forgets, how many other methods also forget
forgot = {}
for (m, o, sd), (d, s) in runs.items():
    order, last = s['order'], len(s['order'])
    for k, ds in enumerate(order[:-1], 1):
        a, z = M(d, s, k, ds).align(M(d, s, last, ds), join='inner')
        forgot[(m, o, sd, ds)] = set(a.index[(a > 0) & (z <= 0)])
rows = []
for a in METHODS:
    row = {'Forgotten by': LABEL[a], 'Pairs forgotten': sum(len(f) for k, f in forgot.items() if k[0] == a)}
    for b in METHODS:
        hit = tot = 0
        for k, f in forgot.items():
            if k[0] == a and (b,) + k[1:] in forgot:
                hit += len(f & forgot[(b,) + k[1:]])
                tot += len(f)
        row[f'also by {LABEL[b]} (%)'] = 100 * hit / max(tot, 1)
    rows.append(row)
show(pd.DataFrame(rows), 'A4_shared_forgetting', 'Of the pairs the row method forgets, % also forgotten by the column method (same order, seed, behavior)')

## Table A5. Do methods replay the same training pairs?

In [ ]:
# A5. Overlap of the training pairs each method replays (Jaccard, same order, seed and stage)
sets = {}
for (m, o, sd), (d, s) in runs.items():
    for k in range(2, len(s['order']) + 1):
        p = sdir(d, s, k) / 'replay_log.csv'
        if p.exists():
            lg = pd.read_csv(p)
            if len(lg):
                sets[(m, o, sd, k)] = set(lg['id'])
rep = [m for m in METHODS if m != 'none']
rows = []
for a in rep:
    row = {'Method': LABEL[a]}
    for b in rep:
        v = [len(sets[(a,) + k[1:]] & sets[(b,) + k[1:]]) / len(sets[(a,) + k[1:]] | sets[(b,) + k[1:]])
             for k in sets if k[0] == a and (b,) + k[1:] in sets]
        row[LABEL[b]] = np.mean(v) if v else np.nan
    rows.append(row)
show(pd.DataFrame(rows), 'A5_replayed_pair_overlap', 'Overlap (Jaccard) of the training pairs two methods replay in the same order, seed and stage')

## Table A6. Forgetting by learned margin, and recovery

In [ ]:
# A6. Forgetting by learned-margin quartile and recovery (per method)
rows = []
for (m, o, sd), (d, s) in runs.items():
    order, last = s['order'], len(s['order'])
    for k, ds in enumerate(order[:-1], 1):
        a, z = M(d, s, k, ds).align(M(d, s, last, ds), join='inner')
        kn = a[a > 0]
        q = pd.qcut(kn.rank(method='first'), 4, labels=[1, 2, 3, 4])
        r = {'method': m}
        for qq in [1, 2, 3, 4]:
            r[f'Q{qq}'] = 100 * (z.reindex(kn.index[q == qq]) <= 0).mean()
        un = a <= 0
        r['recovery'] = 100 * ((z > 0) & un).sum() / max(un.sum(), 1)
        r['forget'] = 100 * ((z <= 0) & (a > 0)).sum() / max((a > 0).sum(), 1)
        rows.append(r)
t = pd.DataFrame(rows).groupby('method').mean().reindex(METHODS).reset_index()
t['method'] = t['method'].map(LABEL)
t = t.rename(columns={'method': 'Method', 'Q1': 'Forgotten, weakest quarter (%)', 'Q2': 'Q2 (%)', 'Q3': 'Q3 (%)',
                      'Q4': 'Forgotten, strongest quarter (%)', 'forget': 'Forgotten (% of learned)',
                      'recovery': 'Recovered (% of not learned)'})
show(t, 'A6_margin_quartiles_and_recovery', 'Forgetting by how strongly a pair was learned, and recovery of pairs not learned')

## Table A7. A shared core of forgotten pairs (Figure 2 of the report)

In [ ]:
# A7. Shared core (Figure 2 of the report). In each order, seed and earlier behavior, take the pairs that EVERY
# method had right just after learning, and count how many methods forget each one by the end.
rows = []
for o, sd in sorted({k[1:] for k in runs}):
    if not all((m, o, sd) in runs for m in METHODS):
        continue
    order = runs[(METHODS[0], o, sd)][1]['order']
    last = len(order)
    for k, ds in enumerate(order[:-1], 1):
        learned, final = {}, {}
        for m in METHODS:
            d, s = runs[(m, o, sd)]
            learned[m], final[m] = M(d, s, k, ds), M(d, s, last, ds)
        known = set.intersection(*[set(learned[m].index[learned[m] > 0]) for m in METHODS])
        known = sorted(known & set.intersection(*[set(final[m].index) for m in METHODS]))
        fg = {m: set(final[m].reindex(known).index[final[m].reindex(known) <= 0]) for m in METHODS}
        core = set.intersection(*fg.values())
        chance = len(known) * np.prod([len(fg[m]) / max(len(known), 1) for m in METHODS])
        for m in METHODS:
            others = set.union(*[fg[x] for x in METHODS if x != m])
            rows.append({'method': m, 'total': len(fg[m]), 'core': len(core), 'only': len(fg[m] - others),
                         'chance_core': chance})
t = pd.DataFrame(rows).groupby('method')[['total', 'core', 'only', 'chance_core']].sum().reindex(METHODS)
t['some'] = t['total'] - t['core'] - t['only']
t = t.reset_index()
t['method'] = t['method'].map(LABEL)
t = t[['method', 'total', 'core', 'some', 'only', 'chance_core']].rename(columns={
    'method': 'Method', 'total': 'Pairs forgotten', 'core': 'Forgotten by all 5', 'some': 'Also by some others',
    'only': 'Only this method', 'chance_core': 'All 5 expected by chance'})
show(t, 'A7_shared_core', 'Pairs every method had learned: forgotten by all 5, by some, or only by this method (summed over runs)')


## Figures

In [ ]:
a2 = TABLES['A2_stage_effect_by_length'].set_index('Stage being trained')
fa2 = pd.DataFrame({'short_flip': a2['chosen shorter: flipped_pct'], 'long_flip': a2['chosen longer: flipped_pct']}).reindex(['Helpful', 'Safe', 'Quality'])
a3 = TABLES['A3_safety_vs_quality_after'].pivot(index='Quality after Safety', columns='Safety pairs', values='Forgotten (% of learned)')
fa3 = pd.DataFrame({'short_forgot': a3['chosen shorter'], 'long_forgot': a3['chosen longer']}).reindex([False, True])
FIGURES['FigA1_answer_length'] = fig_length(fa2, fa3)
display(FIGURES['FigA1_answer_length']); plt.close(FIGURES['FigA1_answer_length'])

In [ ]:
a4 = TABLES['A4_shared_forgetting'].set_index('Forgotten by')
a4 = a4[[c for c in a4.columns if c.startswith('also by')]]
a4.index = [k for l in a4.index for k, v in LABEL.items() if v == l]
a4.columns = list(a4.index)
a5 = TABLES['A5_replayed_pair_overlap'].set_index('Method')
a5.index = [k for l in a5.index for k, v in LABEL.items() if v == l]
a5.columns = list(a5.index)
FIGURES['FigA2_shared_forgetting'] = fig_overlap(a5, a4)
display(FIGURES['FigA2_shared_forgetting']); plt.close(FIGURES['FigA2_shared_forgetting'])

In [ ]:
a6 = TABLES['A6_margin_quartiles_and_recovery'].set_index('Method')
a6.index = [k for l in a6.index for k, v in LABEL.items() if v == l]
a6 = a6.rename(columns={'Forgotten, weakest quarter (%)': 'Q1', 'Q2 (%)': 'Q2', 'Q3 (%)': 'Q3', 'Forgotten, strongest quarter (%)': 'Q4',
                        'Forgotten (% of learned)': 'forget', 'Recovered (% of not learned)': 'recovery'})
FIGURES['FigA3_learned_margin'] = fig_margin(a6)
display(FIGURES['FigA3_learned_margin']); plt.close(FIGURES['FigA3_learned_margin'])

In [ ]:
def fig_core(t7):
    """Stacked bars: of the pairs each method forgets, how many are the shared core, shared with some, or unique."""
    y = np.arange(len(t7))[::-1]
    fig, ax = plt.subplots(figsize=(FULL_W, 2.3))
    parts = [('Forgotten by all 5', '#b3261e', 'Forgotten by all 5 methods'),
             ('Also by some others', '#ee8b84', 'Also forgotten by some other methods'),
             ('Only this method', '#c9c8c3', 'Forgotten by this method only')]
    left = np.zeros(len(t7))
    for col, c, lab in parts:
        v = t7[col].values.astype(float)
        ax.barh(y, v, left=left, color=c, height=0.62, label=lab, zorder=2)
        for yi, l, vi in zip(y, left, v):
            if vi >= 0.065 * t7['Pairs forgotten'].max():
                ax.text(l + vi / 2, yi, f'{vi:.0f}', ha='center', va='center', fontsize=8.5,
                        color='white' if col == 'Forgotten by all 5' else INK)
        left += v
    for yi, tot in zip(y, t7['Pairs forgotten']):
        ax.text(tot + 0.01 * t7['Pairs forgotten'].max(), yi, f'{tot:.0f}', va='center', fontsize=8.5, color=INK2)
    ax.set_yticks(y, t7['Method'])
    ax.tick_params(axis='y', length=0)
    ax.set_xlabel('Validation pairs forgotten, summed over 8 runs (pairs every method had learned)')
    ax.set_xlim(0, t7['Pairs forgotten'].max() * 1.1)
    ax.grid(axis='x', color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    fig.legend(*ax.get_legend_handles_labels(), loc='outside upper center', ncol=3, handlelength=1.2, columnspacing=1.2)
    return fig


t7 = TABLES['A7_shared_core']
print(f"Shared core: {t7['Forgotten by all 5'].iloc[0]:.0f} pairs forgotten by all 5 methods "
      f"(chance {t7['All 5 expected by chance'].iloc[0]:.1f})")
FIGURES['Fig2_shared_core'] = fig_core(t7)
display(FIGURES['Fig2_shared_core']); plt.close(FIGURES['Fig2_shared_core'])


## Save (run when you want the files)

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for name, df in TABLES.items():
    df.to_csv(OUT / f'{name}.csv', index=False)
for name, fig in FIGURES.items():
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight', pad_inches=0.04)
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight', pad_inches=0.04)
print('saved to', OUT)